# Customer Churn Prediction with SQL & Machine Learning

## Project Overview
This project analyzes customer behavior and predicts churn using behavioral, subscription, and engagement data.

The workflow includes:
- Loading raw customer data into a SQLite database
- Performing exploratory SQL analysis
- Engineering features from behavioral metrics
- Building machine learning models to predict churn

Dataset size: 125,000 customers

## Load Dataset and Create SQLite Database

We load the churn dataset and store it in a SQLite database to enable SQL-based exploration and feature analysis.

In [1]:
import sqlite3
import pandas as pd

# Load your CSV into a SQLite database
conn = sqlite3.connect("churn.db")

train = pd.read_csv("train.csv")
train.to_sql("customers", conn, if_exists="replace", index=False)

print("Database created successfully")
print(pd.read_sql("SELECT COUNT(*) as total_rows FROM customers", conn))

Database created successfully
   total_rows
0      125000


## Exploratory SQL Analysis

We explore key churn drivers using SQL queries. These queries analyze how churn relates to:

- Subscription type
- Customer service inquiries
- User engagement behavior
- Payment plan and payment method

The following queries are saved as reusable SQL files for further analysis and reporting.

In [9]:
import os
os.makedirs("sql", exist_ok=True)

# Query 1
with open("sql/churn_by_subscription_type.sql", "w") as f:
    f.write("""-- Churn rate by subscription type
-- Free users churn at 79% vs 34% for Premium/Family, showing paid plans drive retention

SELECT 
    subscription_type,
    COUNT(*) AS total_customers,
    SUM(churned) AS churned_customers,
    ROUND(AVG(churned) * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY subscription_type
ORDER BY churn_rate_pct DESC;
""")

# Query 2
with open("sql/churn_by_service_inquiries.sql", "w") as f:
    f.write("""-- Churn rate by customer service inquiry level
-- High inquiry users churn at 74% vs 29% for Low — nearly 3x difference

SELECT
    customer_service_inquiries,
    COUNT(*) AS total_customers,
    SUM(churned) AS churned_customers,
    ROUND(AVG(churned) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(weekly_hours), 2) AS avg_weekly_hours,
    ROUND(AVG(num_subscription_pauses), 2) AS avg_pauses
FROM customers
GROUP BY customer_service_inquiries
ORDER BY churn_rate_pct DESC;
""")

# Query 3
with open("sql/churned_vs_retained_behavior.sql", "w") as f:
    f.write("""-- Behavioral comparison: churned vs retained customers
-- Churned users spend 28% less time on platform and skip 20% more songs

SELECT
    churned,
    COUNT(*) AS total_customers,
    ROUND(AVG(-signup_date), 1) AS avg_tenure_days,
    ROUND(AVG(weekly_hours), 2) AS avg_weekly_hours,
    ROUND(AVG(song_skip_rate), 3) AS avg_skip_rate,
    ROUND(AVG(num_playlists_created), 2) AS avg_playlists,
    ROUND(AVG(notifications_clicked), 2) AS avg_notifications
FROM customers
GROUP BY churned;
""")

# Query 4
with open("sql/churn_by_payment.sql", "w") as f:
    f.write("""-- Churn rate by payment plan and method
-- Churn is consistent across all payment combinations (50-52%)
-- suggesting payment type is not a driver of churn

SELECT
    payment_plan,
    payment_method,
    COUNT(*) AS total_customers,
    ROUND(AVG(churned) * 100, 2) AS churn_rate_pct
FROM customers
GROUP BY payment_plan, payment_method
ORDER BY churn_rate_pct DESC;
""")

print("SQL files saved:")
for f in os.listdir("sql"):
    print(f" - sql/{f}")

SQL files saved:
 - sql/churn_by_subscription_type.sql
 - sql/churn_by_payment.sql
 - sql/churn_by_service_inquiries.sql
 - sql/churned_vs_retained_behavior.sql
